# TÉCNICA SUPER RESOLUTION CON ESCALA 4

In [ ]:
import os
import glob
import random
import time
from pathlib import Path
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import StepLR

# Métricas
from skimage.metrics import peak_signal_noise_ratio as psnr_skimage
from skimage.metrics import structural_similarity as ssim_skimage
import lpips

# Visualización 
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Rutas del dataset 
HR_DIR  = Path('dataset/Training')                  
LR_DIR = Path('dataset/Training_LR_CLAHE')   # carpeta generada con escala 4

# Hiperparámetros 
IMG_H        = 112
IMG_W        = 92
BATCH_SIZE   = 16     
NUM_EPOCHS   = 50
LR_INIT      = 1e-4
SCHED_STEP   = 30     
SCHED_GAMMA  = 0.5
VAL_SPLIT    = 0.15
WEIGHTS_PATH = 'srcnn_orl_best.pth'

print('\n Configuración cargada.')
print(f'   HR_DIR  existe: {HR_DIR.exists()}')
print(f'   LR_DIR  existe: {LR_DIR.exists()}')